In [14]:
import pandas as pd

train = pd.read_csv("train.csv", low_memory=False)
store = pd.read_csv("store.csv")

print(train.shape)
print(store.shape)

(1017209, 9)
(1115, 10)


In [15]:
train.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [16]:
store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [17]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype
---  ------         --------------    -----
 0   Store          1017209 non-null  int64
 1   DayOfWeek      1017209 non-null  int64
 2   Date           1017209 non-null  str  
 3   Sales          1017209 non-null  int64
 4   Customers      1017209 non-null  int64
 5   Open           1017209 non-null  int64
 6   Promo          1017209 non-null  int64
 7   StateHoliday   1017209 non-null  str  
 8   SchoolHoliday  1017209 non-null  int64
dtypes: int64(7), str(2)
memory usage: 80.5 MB


In [18]:
store.isnull().sum()

Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64

In [19]:
print((store["Promo2"] == 0).sum())

544


In [20]:
store["CompetitionDistance"] = store["CompetitionDistance"].fillna(store["CompetitionDistance"].median())

In [21]:
store.isnull().sum()

Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            0
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64

In [24]:
train["Date"] = pd.to_datetime(train["Date"])

In [26]:
print((train["Open"] == 0).sum())   # pehle count dekho
train = train[train["Open"] == 1]

172817


In [27]:
print(train["StateHoliday"].value_counts()) #StateHoliday ke values

StateHoliday
0    843482
a       694
b       145
c        71
Name: count, dtype: int64


In [28]:
df = train.merge(store, on="Store", how="left")
print(df.shape)

(844392, 18)


In [29]:
df["IsStateHoliday"] = (df["StateHoliday"] != "0").astype(int)

In [30]:
print(train["Date"].min(), train["Date"].max())

2013-01-01 00:00:00 2015-07-31 00:00:00


In [31]:
p2 = store[store["Promo2"] == 1].copy()

p2["Promo2SinceYear"] = p2["Promo2SinceYear"].astype(int)
p2["Promo2SinceWeek"] = p2["Promo2SinceWeek"].astype(int)

# join date (approx): saal ki 1 January + (week - 1) hafte
p2["Promo2Start"] = pd.to_datetime(p2["Promo2SinceYear"].astype(str) + "-01-01") \
                    + pd.to_timedelta((p2["Promo2SinceWeek"] - 1) * 7, unit="D")

In [32]:
print(p2["Promo2SinceYear"].value_counts().sort_index()) #Count according to year

Promo2SinceYear
2009     73
2010     64
2011    128
2012     81
2013    120
2014     95
2015     10
Name: count, dtype: int64


In [33]:
usable = p2[(p2["Promo2Start"] >= "2013-03-01") & (p2["Promo2Start"] <= "2015-06-01")]
print(len(usable))

176
